DH演習2026 · 教材 第51版 · 2026-r51 · 2026-11-19

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第51版
教材ID: 2026-r51
教材日付: 2026-11-19
活動: text_counts
環境: Colab / Jupyter
ソース: a3b1e87c046bdeb87921384a9aa62d2f1f2035ac
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第7回：戯曲全体から発言と人物への言及を数える

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行してください。

資料は芥川龍之介『二人小町』全文（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。5節の人物のタグは教材で加えた注釈です。

## 1. 戯曲全体を取得する

ノートブックは、版を固定したTEIファイルを取得します。`div`は場、`sp`は一つの発言です。

In [ ]:
#@title 操作：本文を取得する
import re
import xml.etree.ElementTree as ET

import pandas as pd
import requests

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
tei = ET.fromstring(requests.get(url).content)
body = tei.findall(f".//{T}body")[0]
divs = body.findall(T + "div")
print("場の数：", len(divs))
print("発言の数：", sum(len(div.findall(T + "sp")) for div in divs))

## 2. 発言を表にする

`who`は発言者の作品内ID、`speaker`は本文に表示される発言者名です。ふりがなは`span`要素（`type="rt"`・`"rp"`）に入っているため、`plain()`は読みと括弧を除いて文字をつなぎます。`rows`は発言を一件ずつ記録したリストです。`speeches`は、そのリストから作った表です。

In [ ]:
def plain(e):
    if e.tag == T + "span" and e.get("type") in ("rt", "rp"):
        return e.tail or ""
    return (e.text or "") + "".join(plain(child) for child in e) + (e.tail or "")

rows = []
for div in divs:
    for n, sp in enumerate(div.findall(T + "sp"), 1):
        speaker, *rest = sp
        rows.append({"act": div.get("n"), "n": n, "who": sp.get("who"), "speaker": speaker.text, "text": "".join(plain(e) for e in rest)})
speeches = pd.DataFrame(rows)
speeches.head(8)

## 3. 集計キーと数える範囲を変える

`group_by`を`"speaker"`にすると表示名別、`"who"`にすると作品内ID別に数えます。変える前に、「小町」の発言がどのキーにまとまり、全発言数がどうなるか予測してください。

元のセルを実行してから`group_by`を変え、再実行してください。表示された合計を、`rows`に入っている全発言数と照合してください。

余裕があれば、選んだ場だけを数えてください。`for`の中で、`speech["act"]`と`act`が違うときは件数を増やす前に`continue`で次の発言に進みます。合計を、二つ目の表の`count`列の合計と照合してください。

In [ ]:
group_by = "speaker"
act = "一"

counts = {}  # type: dict[str | int | None, int]
for speech in rows:
    key = speech[group_by]
    # 初めて出たキーの件数は0から始める
    counts[key] = counts.get(key, 0) + 1

print(counts)
print("合計発言数：", sum(counts.values()))
speeches[speeches["act"] == act].groupby(["who", "speaker"]).size().reset_index(name="count")

## 4. 文字列検索で数える

ノートブックは、一つの検索語`word`を入力どおりの文字列として数えます。数える範囲で件数が変わります。

- **読みを含む本文**：ふりがなが名前の途中に入ると、検索に掛かりません。
- **読みを除いた本文**：発言者名の表示と舞台指示も数えます。
- **発言の中**：発言者名の表示と、発言の外の舞台指示を除きます。発言の途中の舞台指示は残ります。

`word`を「あの人」に変えると、誰を指すかは件数から分かりますか。

In [ ]:
word = "玉造の小町"

print("読みを含む本文：", "".join(body.itertext()).count(word))
print("読みを除いた本文：", plain(body).count(word))
# 正規表現で使う記号も、入力どおりの文字として数える
print("発言の中：", speeches["text"].str.count(re.escape(word)).sum())
speeches[speeches["text"].str.contains(word, regex=False)][["act", "n", "who", "text"]]

## 5. 注釈した言及を数える

文字列検索は、「あの人」が誰を指すかを区別しません。`material`は、第1場の発言14・15・19に人物のタグを付けた注釈案です。固有名には`persName`、代名詞などの参照表現には`rs type="person"`を付け、`ref="#tama"`で人物IDを指します。

`include_references`を`False`にすると`rs`を数えません。同じ3発言を`word`で検索した件数と比べてください。

In [ ]:
material = '''<body xmlns="http://www.tei-c.org/ns/1.0">
<sp xml:id="s1-sp14" who="#messenger"><speaker>使</speaker><p>いや、名前もあなたのように小町と云わなければいけないのです。</p></sp>
<sp xml:id="s1-sp15" who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。</p><stage>（発作的に笑い出しながら）</stage><p><persName ref="#tama">玉造の小町</persName>と云う人がいます。<rs type="person" ref="#tama">あの人</rs>を代りにつれて行って下さい。</p></sp>
<sp xml:id="s1-sp19" who="#ono"><speaker>小町</speaker><stage>（生き生きと）</stage><p>では<rs type="person" ref="#tama">あの人</rs>に行って貰って下さい。<rs type="person" ref="#tama">あの人</rs>はこの世にいるよりも、地獄に住みたいと云っています。誰も逢う人がいないものですから。</p></sp>
</body>'''

include_references = True

mentions = []
for e in ET.fromstring(material).iter():
    # rsはスイッチが有効で、人物を指すときだけ拾う
    if e.tag == T + "persName" or (include_references and e.tag == T + "rs" and e.get("type") == "person"):
        mentions.append({"text": "".join(e.itertext()), "id": e.get("ref", "")[1:]})
print("タグの付いた言及：", pd.DataFrame(mentions)["id"].value_counts().to_dict())
print("文字列検索：", "".join(ET.fromstring(material).itertext()).count(word))
pd.DataFrame(mentions)

## 確認問題

第2場で、表示名が「小町」の発言は誰の発言ですか。3節の`act`を変えて再実行し、`answer`にその発言者のID（`who`の値）と発言の件数を入れてください。

In [ ]:
answer = {"who": "", "count": None}

In [ ]:
#@title 答えを確かめる
if not answer["who"] or answer["count"] is None:
    print("結果：不合格（`answer`に発言者のIDと件数を入れてください）")
elif answer == {"who": "#小野の小町", "count": 11}:
    print("結果：不合格（第1場の結果です。`act`を「二」に変えてください）")
elif answer["who"] != "#玉造の小町":
    print("結果：不合格（3節の二つ目の表で、`speaker`が「小町」の行の`who`を確かめてください）")
elif answer["count"] != 22:
    print("結果：不合格（同じ行の`count`を確かめてください）")
else:
    print("結果：合格")